# 🚀 Huấn Luyện 100 Epochs ResMLP / RepMLP trên CIFAR-100 (Google Colab GPU)

Notebook này hướng dẫn chi tiết cách chạy trọn vẹn **100 epochs** với GPU miễn phí (NVIDIA T4):
- **Thời gian dự kiến**: ~15 - 20 phút cho 100 epochs (nhanh gấp ~25 lần so với CPU Mac).
- **Tính năng**: Tự động tải CIFAR-100, bật Mixed Precision (AMP FP16), tự động lưu checkpoint vào Google Drive.

### Bước 1: Kiểm tra GPU NVIDIA T4
Vào menu **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** trước khi chạy.

In [ ]:
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')

### Bước 2: Kết nối Google Drive
Mount Google Drive để lưu checkpoint (`best_model.pt`) và logs an toàn, không lo mất file khi Colab ngắt kết nối.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Bước 3: Lấy Code dự án vào Colab
Chọn **MỘT** trong 2 cách bên dưới:
- **Cách A (Khuyên dùng)**: Clone trực tiếp từ GitHub.
- **Cách B**: Nếu bạn upload file zip thư mục dự án lên Colab (kéo thả vào thanh sidebar bên trái).

In [ ]:
# Cách A: Clone từ GitHub repo của team
import os
if not os.path.exists('/content/DebugTeam_Coding'):
    !git clone https://github.com/AIVIETNAM-AIO-MyNguyen/DebugTeam_Coding.git /content/DebugTeam_Coding
%cd /content/DebugTeam_Coding

# Cách B: Nếu bạn nén zip từ máy và upload lên Colab tên DebugTeam_Coding-main.zip, bỏ comment 2 dòng dưới:
# !unzip -q /content/DebugTeam_Coding-main.zip -d /content/
# %cd /content/DebugTeam_Coding-main

### Bước 4: Cài đặt thư viện phụ thuộc

In [ ]:
!pip install -q timm
!pip install -q -e .

### Bước 5: Chạy Huấn Luyện 100 Epochs (ResMLP)
Cấu hình `configs/cifar100_res_mlp.yaml` đã được thiết lập sẵn:
- `epochs: 100`
- `batch_size: 128`
- `use_amp: true` (Mixed Precision FP16 trên T4 Tensor Cores)
- Tự động tải CIFAR-100

In [ ]:
!python scripts/run_train.py --config configs/cifar100_res_mlp.yaml

### Bước 6: Sao lưu Checkpoints & Logs sang Google Drive
Sao lưu toàn bộ kết quả để tải về máy hoặc lưu trữ lâu dài.

In [ ]:
!mkdir -p /content/drive/MyDrive/MLP_Experiments_Runs
!cp -r runs/* /content/drive/MyDrive/MLP_Experiments_Runs/
print('✅ Đã sao lưu toàn bộ kết quả vào Google Drive: /content/drive/MyDrive/MLP_Experiments_Runs/')

### Bước 7: Vẽ biểu đồ Loss & Accuracy sau 100 Epochs

In [ ]:
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt

metric_files = list(Path('runs').rglob('metrics.json'))
if metric_files:
    latest_metric = sorted(metric_files, key=os.path.getmtime)[-1]
    print(f'Đang đọc kết quả từ: {latest_metric}')
    with open(latest_metric, 'r') as f:
        data = json.load(f)
    
    history = data.get('history', [])
    if history:
        epochs = [h['epoch'] for h in history]
        train_loss = [h['train_loss'] for h in history]
        val_loss = [h['val_loss'] for h in history]
        val_top1 = [h['val_top1'] for h in history]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(epochs, train_loss, label='Train Loss', color='tab:blue')
        ax1.plot(epochs, val_loss, label='Val Loss', color='tab:orange')
        ax1.set_title('CrossEntropy Loss qua 100 Epochs')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.grid(True)
        ax1.legend()

        ax2.plot(epochs, val_top1, label='Val Top-1 Acc', color='tab:green', linewidth=2)
        ax2.set_title(f'Validation Accuracy (Đỉnh: {max(val_top1):.2f}%)')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy (%)')
        ax2.grid(True)
        ax2.legend()
        plt.tight_layout()
        plt.savefig('runs/cifar100_100epochs_plot.png', dpi=150)
        plt.show()
else:
    print('Chưa tìm thấy metrics.json. Hãy chắc chắn quá trình train đã hoàn tất.')

### Bước 8 (Tùy chọn): Kiểm tra Locality Injection & Reparameterization trên GPU
Chuyển đổi các nhánh Convolution (1x1, 3x3) thành 1 ma trận Linear duy nhất để suy luận tốc độ cao:

In [ ]:
import torch
from src.models.res_mlp import ResMLP

# Khởi tạo ResMLP kèm LocalPerceptron
model = ResMLP(in_channels=3, image_size=32, patch_size=4, num_features=128, expansion_factor=2, num_layers=8, num_classes=100, use_local_perceptron=True).cuda()

# Forward thử nghiệm trước khi fuse
x = torch.randn(8, 3, 32, 32, device='cuda')
out_train = model(x)

# Thực hiện Locality Injection (Structural Re-parameterization)
model.locality_injection()
out_deploy = model(x)

diff = (out_train - out_deploy).abs().max().item()
print(f'Độ sai lệch tối đa giữa bản Train và bản Deploy (Fused): {diff:.2e}')
assert diff < 1e-4, 'Re-parameterization không khớp!'
print('🎉 Structural Re-parameterization thành công 100% trên Colab GPU!')